## 1. Load Train/Test Split

In [1]:
import pandas as pd

X_train = pd.read_csv("../data/splits/X_train.csv")
X_test = pd.read_csv("../data/splits/X_test.csv")
y_train = pd.read_csv("../data/splits/y_train.csv")
y_test = pd.read_csv("../data/splits/y_test.csv")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (21367, 1)
X_test shape: (5342, 1)
y_train shape: (21367, 1)
y_test shape: (5342, 1)


In [2]:
X_train.head()

,headline
0,fbi raids fridge
1,old el paso introduces emergency taco kit
2,larry kudlow 'leaning' toward senate run in co...
3,"'everything's $10,000' chain goes out of business"
4,what this ceo did proves that introverts make ...


## 2. Feature Engineering

In [4]:
!pip install nltk

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 11.4 MB/s  0:00:00

   ---------------------------------------- 0/4 [tqdm]
   ---------- ----------------------------- 1/4 [regex]
   ---------- ----------------------------- 1/4 [regex]
   -------------------- ------------------- 2/4 [click]
   -------------------- ------------------- 2/4 [click]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------------ --------- 3/4 [nltk]
   ------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download VADER lexicon once
nltk.download("vader_lexicon")

sia = SentimentIntensityAnalyzer()

def count_all_caps(text: str) -> int:
    return sum(1 for word in str(text).split() if word.isupper() and len(word) > 1)

def count_punctuation(text: str, mark: str) -> int:
    return str(text).count(mark)

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Length-based feature
    df["length"] = df["headline"].apply(len)
    
    # Word count
    df["word_count"] = df["headline"].apply(lambda x: len(str(x).split()))
    
    # Punctuation features
    df["exclamation_count"] = df["headline"].apply(lambda x: count_punctuation(x, "!"))
    df["question_count"] = df["headline"].apply(lambda x: count_punctuation(x, "?"))
    
    # All-caps feature
    df["all_caps_count"] = df["headline"].apply(count_all_caps)
    
    # VADER sentiment
    df["vader_neg"] = df["headline"].apply(lambda x: sia.polarity_scores(str(x))["neg"])
    df["vader_neu"] = df["headline"].apply(lambda x: sia.polarity_scores(str(x))["neu"])
    df["vader_pos"] = df["headline"].apply(lambda x: sia.polarity_scores(str(x))["pos"])
    df["vader_compound"] = df["headline"].apply(lambda x: sia.polarity_scores(str(x))["compound"])
    
    return df

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\denze\AppData\Roaming\nltk_data...


In [7]:
# Apply feature engineering to training and test sets to create new numeric features

X_train_fe = feature_engineering(X_train)
X_test_fe = feature_engineering(X_test)

X_train_fe.head()

,headline,length,word_count,exclamation_count,question_count,all_caps_count,vader_neg,vader_neu,vader_pos,vader_compound
0,fbi raids fridge,16,3,0,0,0,0.000,1.000,0.000,0.0000
1,old el paso introduces emergency taco kit,41,7,0,0,0,0.302,0.698,0.000,-0.3818
2,larry kudlow 'leaning' toward senate run in co...,55,8,0,0,0,0.000,1.000,0.000,0.0000
3,"'everything's $10,000' chain goes out of business",49,7,0,0,0,0.000,1.000,0.000,0.0000
4,what this ceo did proves that introverts make ...,59,10,0,0,0,0.000,0.687,0.313,0.6249


# 3.TF-IDF vectorization

In [8]:
# Convert headline text into TF-IDF vectors so the models can use numerical text features

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train["headline"])
X_test_tfidf = tfidf.transform(X_test["headline"])

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (21367, 5000)
X_test_tfidf shape: (5342, 5000)


In [9]:
# Output Check(Expected: rows = number of headlines/columns = number of TF-IDF features, up to 5000

print(type(X_train_tfidf))
print(X_train_tfidf.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(21367, 5000)


## 5. Logistic Regression (TF-IDF Features)

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Train model
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test_tfidf)

# Evaluate
print("Logistic Regression Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))

Logistic Regression Results:
Accuracy: 0.8341445151628604
Precision: 0.8217909131010146
Recall: 0.794456289978678
F1 Score: 0.8078924544666088


c:\Users\denze\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


## 6. Naive Bayes (TF-IDF Features)

In [11]:
from sklearn.naive_bayes import MultinomialNB

# Train model
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Predict
y_pred_nb = nb_model.predict(X_test_tfidf)

# Evaluate
print("Naive Bayes Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall:", recall_score(y_test, y_pred_nb))
print("F1 Score:", f1_score(y_test, y_pred_nb))

Naive Bayes Results:
Accuracy: 0.8255335080494197
Precision: 0.8548468106479156
Recall: 0.7257995735607676
F1 Score: 0.7850553505535055


c:\Users\denze\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


## 7. Model Comparison

In [12]:
results = {
    "Model": ["Logistic Regression", "Naive Bayes"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_nb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_nb)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_nb)
    ]
}

import pandas as pd
pd.DataFrame(results)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.834145,0.821791,0.794456,0.807892
1,Naive Bayes,0.825534,0.854847,0.725800,0.785055
